# EDA Notebook 4 v2 — Core Cross-Dataset Audit and Freeze

**Thesis:** *An Ethical Risk Assessment of Sentiment Analysis Models in Social Media Monitoring with a Focus on Bias, Inclusivity, and Misclassification Patterns*  
**Student:** Abhishek Tomar | PN1196973 | LJMU

## Purpose
This notebook runs **after EDA_01 TweetEval, EDA_02 SemEval-2014 and EDA_03 Sentiment140 have been frozen**.

It does **not** redefine linguistic groups, tune thresholds, pool subgroup counts to manufacture eligibility, or train/select any model.

It consolidates the three approved datasets into one evidence map:
1. verify authoritative EDA outputs;
2. preserve each dataset's distinct research role;
3. compare task and label-space compatibility;
4. compare subgroup coverage under the frozen TweetEval rules;
5. state which cross-dataset claims are permitted;
6. save a machine-readable core-dataset EDA freeze for Chapter 3.

## Cell 1 — Imports and locate authoritative Results root

In [ ]:
!pip install -q pyarrow

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import glob, json
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

matches = glob.glob('/content/drive/MyDrive/**/Results/TweetEval/tweeteval_freeze_config.json', recursive=True)
if not matches:
    raise FileNotFoundError('Run EDA_01_TweetEval_FINAL_v2 first.')

config_path = Path(matches[0])
RESULTS_ROOT = config_path.parent.parent
OUTDIR = RESULTS_ROOT / 'CrossDataset'
OUTDIR.mkdir(parents=True, exist_ok=True)

print('Results root:', RESULTS_ROOT)
print('Output directory:', OUTDIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results root: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results
Output directory: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/CrossDataset


## Cell 2 — Load frozen outputs from EDA_01–03

In [ ]:
TW_DIR = RESULTS_ROOT / 'TweetEval'
SE_DIR = RESULTS_ROOT / 'SemEval2014'
S140_DIR = RESULTS_ROOT / 'Sentiment140'

paths = {
    'tw_df': TW_DIR/'df_tweeteval_final.parquet',
    'tw_freeze': TW_DIR/'tweeteval_freeze_config.json',
    'tw_integrity': TW_DIR/'tweeteval_integrity_summary.json',
    'se_df': SE_DIR/'df_semeval2014_final.parquet',
    'se_role': SE_DIR/'semeval_role_freeze.json',
    'se_integrity': SE_DIR/'semeval_integrity_summary.json',
    's140_df': S140_DIR/'df_sentiment140_eval_sample_final.parquet',
    's140_role': S140_DIR/'sentiment140_role_freeze.json',
    's140_integrity': S140_DIR/'sentiment140_integrity_summary.json',
}
missing = [k for k,p in paths.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing authoritative outputs: {missing}')

with open(paths['tw_freeze'], encoding='utf-8') as f: tw_freeze=json.load(f)
with open(paths['tw_integrity'], encoding='utf-8') as f: tw_integrity=json.load(f)
with open(paths['se_role'], encoding='utf-8') as f: se_role=json.load(f)
with open(paths['se_integrity'], encoding='utf-8') as f: se_integrity=json.load(f)
with open(paths['s140_role'], encoding='utf-8') as f: s140_role=json.load(f)
with open(paths['s140_integrity'], encoding='utf-8') as f: s140_integrity=json.load(f)

tw = pd.read_parquet(paths['tw_df'])
se = pd.read_parquet(paths['se_df'])
s140 = pd.read_parquet(paths['s140_df'])

print('TweetEval:', f'{len(tw):,}')
print('SemEval aspect rows:', f'{len(se):,}')
print('Sentiment140 evaluation sample:', f'{len(s140):,}')

TweetEval: 59,899
SemEval aspect rows: 7,694
Sentiment140 evaluation sample: 100,000


## Cell 3 — Freeze-consistency assertions

In [ ]:
MIN_N = int(tw_freeze['minimum_n_for_stable_quantitative_interpretation'])

assert MIN_N == 30
assert float(tw_freeze['emoji_heavy']['threshold']) == 0.05
assert float(tw_freeze['slang_heavy']['threshold']) == 0.10
assert int(tw_freeze['slang_heavy']['lexicon_size']) == 51
assert int(tw_freeze['reference_unmarked']['minimum_word_count']) == 8
assert tw_freeze['priority_rule'] == [
    'sarcasm-indicated','emoji-heavy','slang-heavy','reference-unmarked','other'
]

assert se_role['role_in_thesis'] == 'domain_task_shift_stress_test'
assert se_role['used_for_training'] is False
assert se_role['used_for_model_selection'] is False
assert se_role['clean_same_task_external_validation'] is False

assert s140_role['role_in_thesis'] == 'binary_external_robustness'
assert s140_role['used_for_training'] is False
assert s140_role['used_for_model_selection'] is False
assert s140_role['label_space_shift_from_tweeteval'] is True

assert s140_integrity['evaluation_sample_rows_with_source_like_emoticon_removed'] == 864
assert s140_integrity['evaluation_sample_empty_after_stripping'] == 0

print('PASS — frozen roles, thresholds and Sentiment140 leakage bookkeeping are consistent.')

PASS — frozen roles, thresholds and Sentiment140 leakage bookkeeping are consistent.


## Cell 4 — Dataset role matrix

In [ ]:
role_matrix = pd.DataFrame([
    {
        'dataset':'TweetEval',
        'unit':'tweet/post',
        'label_space':'negative / neutral / positive',
        'role':'primary sentiment dataset',
        'training':'official train only',
        'selection':'validation/CV only',
        'final_evaluation':'official held-out test once after model lock',
        'subgroup_role':'primary linguistic subgroup audit'
    },
    {
        'dataset':'Sentiment140',
        'unit':'tweet/post',
        'label_space':'negative / positive',
        'role':'binary external robustness',
        'training':'no',
        'selection':'no',
        'final_evaluation':'fixed balanced 100k unique-text sample',
        'subgroup_role':'external subgroup robustness where N >= 30'
    },
    {
        'dataset':'SemEval-2014',
        'unit':'aspect–sentiment pair',
        'label_space':'negative / neutral / positive at aspect level',
        'role':'domain/task-shift stress test',
        'training':'no',
        'selection':'no',
        'final_evaluation':'official test only',
        'subgroup_role':'coverage audit; no focal fairness-transfer claim'
    },
])
print(role_matrix.to_string(index=False))

     dataset                  unit                                   label_space                          role            training          selection                             final_evaluation                                    subgroup_role
   TweetEval            tweet/post                 negative / neutral / positive     primary sentiment dataset official train only validation/CV only official held-out test once after model lock                primary linguistic subgroup audit
Sentiment140            tweet/post                           negative / positive    binary external robustness                  no                 no       fixed balanced 100k unique-text sample       external subgroup robustness where N >= 30
SemEval-2014 aspect–sentiment pair negative / neutral / positive at aspect level domain/task-shift stress test                  no                 no                           official test only coverage audit; no focal fairness-transfer claim


## Cell 5 — Task and label-space compatibility

In [ ]:
comparability = pd.DataFrame([
    {
        'source':'TweetEval',
        'target':'TweetEval test',
        'task_compatibility':'same task',
        'label_compatibility':'same 3-class space',
        'permitted_interpretation':'primary final performance and subgroup evidence'
    },
    {
        'source':'TweetEval',
        'target':'Sentiment140 evaluation sample',
        'task_compatibility':'tweet sentiment, different label construction',
        'label_compatibility':'binary target; explicit label-space shift',
        'permitted_interpretation':'binary external robustness only; raw 3-class macro-F1 not directly comparable'
    },
    {
        'source':'TweetEval',
        'target':'SemEval-2014 test',
        'task_compatibility':'domain + task/label-unit shift',
        'label_compatibility':'aspect-level polarity rather than ordinary sentence-level sentiment',
        'permitted_interpretation':'domain/task-shift stress evidence only'
    },
])
print(comparability.to_string(index=False))

   source                         target                            task_compatibility                                                 label_compatibility                                                      permitted_interpretation
TweetEval                 TweetEval test                                     same task                                                  same 3-class space                               primary final performance and subgroup evidence
TweetEval Sentiment140 evaluation sample tweet sentiment, different label construction                           binary target; explicit label-space shift binary external robustness only; raw 3-class macro-F1 not directly comparable
TweetEval              SemEval-2014 test                domain + task/label-unit shift aspect-level polarity rather than ordinary sentence-level sentiment                                        domain/task-shift stress evidence only


## Cell 6 — Frozen subgroup coverage

In [ ]:
FOCAL = ['emoji-heavy','slang-heavy','sarcasm-indicated','reference-unmarked']

def sg_col(df):
    for c in ['subgroup','subgroup_primary']:
        if c in df.columns:
            return c
    raise KeyError('No subgroup column found')

tw_sg, se_sg, s140_sg = sg_col(tw), sg_col(se), sg_col(s140)

tw_test = tw[tw['split'].astype(str).eq('test')].copy()
tw_cov = tw_test[tw_test[tw_sg].isin(FOCAL)].groupby(tw_sg, observed=True).size().to_dict()

s140_cov = s140[s140[s140_sg].isin(FOCAL)].groupby(s140_sg, observed=True).size().to_dict()

se_test = se[se['split'].astype(str).eq('test')].copy()
if 'text_norm' not in se_test.columns:
    se_test['text_norm'] = (
        se_test['text'].astype(str).str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
    )
se_unique = se_test.drop_duplicates(['domain','split','text_norm'])
se_domain_cov = (
    se_unique[se_unique[se_sg].isin(FOCAL)]
    .groupby(['domain',se_sg], observed=True)
    .size().rename('n').reset_index()
)

rows=[]
for sg in FOCAL:
    n=int(tw_cov.get(sg,0))
    rows.append({
        'dataset':'TweetEval test','subgroup':sg,'n':n,
        'eligibility_unit':'test rows','N>=30':n>=MIN_N,
        'interpretation':'quantitative' if n>=MIN_N else 'descriptive/exploratory only'
    })
    n=int(s140_cov.get(sg,0))
    rows.append({
        'dataset':'Sentiment140 eval sample','subgroup':sg,'n':n,
        'eligibility_unit':'sample rows','N>=30':n>=MIN_N,
        'interpretation':'quantitative external robustness' if n>=MIN_N else 'descriptive/exploratory only'
    })
    for domain in ['laptop','restaurant']:
        part=se_domain_cov[(se_domain_cov['domain']==domain)&(se_domain_cov[se_sg]==sg)]
        n=int(part['n'].iloc[0]) if len(part) else 0
        rows.append({
            'dataset':f'SemEval {domain} test','subgroup':sg,'n':n,
            'eligibility_unit':'unique sentence within domain/test','N>=30':n>=MIN_N,
            'interpretation':'coverage only; task-shift limitation remains' if n>=MIN_N else 'insufficient for focal subgroup transfer'
        })

coverage = pd.DataFrame(rows)
print(coverage.to_string(index=False))

                 dataset           subgroup     n                   eligibility_unit  N>=30                               interpretation
          TweetEval test        emoji-heavy   710                          test rows   True                                 quantitative
Sentiment140 eval sample        emoji-heavy    72                        sample rows   True             quantitative external robustness
     SemEval laptop test        emoji-heavy     0 unique sentence within domain/test  False     insufficient for focal subgroup transfer
 SemEval restaurant test        emoji-heavy     0 unique sentence within domain/test  False     insufficient for focal subgroup transfer
          TweetEval test        slang-heavy    60                          test rows   True                                 quantitative
Sentiment140 eval sample        slang-heavy  2795                        sample rows   True             quantitative external robustness
     SemEval laptop test        slang-hea

## Cell 7 — Subgroup-transfer decision

In [ ]:
transfer_decision = pd.DataFrame([
    {
        'dataset':'TweetEval',
        'emoji-heavy':'quantitative',
        'slang-heavy':'quantitative',
        'sarcasm-indicated':'exploratory only',
        'reference-unmarked':'quantitative',
        'decision':'PRIMARY subgroup evidence'
    },
    {
        'dataset':'Sentiment140',
        'emoji-heavy':'quantitative',
        'slang-heavy':'quantitative',
        'sarcasm-indicated':'quantitative',
        'reference-unmarked':'quantitative',
        'decision':'binary external subgroup robustness; label-space qualification required'
    },
    {
        'dataset':'SemEval-2014',
        'emoji-heavy':'no usable coverage',
        'slang-heavy':'insufficient',
        'sarcasm-indicated':'insufficient',
        'reference-unmarked':'ample but not a focal contrast by itself',
        'decision':'no linguistic fairness-transfer claim; aggregate/domain-task stress test only'
    },
])
print(transfer_decision.to_string(index=False))

     dataset        emoji-heavy  slang-heavy sarcasm-indicated                       reference-unmarked                                                                      decision
   TweetEval       quantitative quantitative  exploratory only                             quantitative                                                     PRIMARY subgroup evidence
Sentiment140       quantitative quantitative      quantitative                             quantitative       binary external subgroup robustness; label-space qualification required
SemEval-2014 no usable coverage insufficient      insufficient ample but not a focal contrast by itself no linguistic fairness-transfer claim; aggregate/domain-task stress test only


## Cell 8 — Sentiment140 leakage-control verification

In [ ]:
print('Full corpus source-like emoticon rows:', f"{s140_integrity['source_like_emoticon_rows']:,}")
print('Evaluation sample rows with source-like emoticon removed:',
      f"{s140_integrity['evaluation_sample_rows_with_source_like_emoticon_removed']:,}")
print('Evaluation sample rows with whitespace normalized:',
      f"{s140_integrity['evaluation_sample_rows_with_whitespace_normalized']:,}")
print('Evaluation sample empty after stripping:',
      s140_integrity['evaluation_sample_empty_after_stripping'])

assert s140_integrity['evaluation_sample_rows_with_source_like_emoticon_removed'] == 864
assert s140_integrity['evaluation_sample_empty_after_stripping'] == 0

print('PASS — emoticon removal is separated from ordinary whitespace normalization.')

Full corpus source-like emoticon rows: 14,299
Evaluation sample rows with source-like emoticon removed: 864
Evaluation sample rows with whitespace normalized: 99,657
Evaluation sample empty after stripping: 0
PASS — emoticon removal is separated from ordinary whitespace normalization.


## Cell 9 — Core methodological decisions

In [ ]:
decisions = [
    ('Primary dataset', 'TweetEval only for training/model selection/final held-out test.'),
    ('Test-set discipline', 'No model, feature, threshold or subgroup definition may be selected from TweetEval test performance.'),
    ('Linguistic tagging', 'Frozen TweetEval rules are reused unchanged on external datasets.'),
    ('Subgroup eligibility', 'N >= 30 is assessed inside the relevant dataset/split; counts are never pooled across corpora.'),
    ('Sentiment140', 'Binary external robustness only; fixed 100k balanced unique-text sample; text_eval removes source-like emoticons.'),
    ('SemEval-2014', 'Domain/task-shift stress test only; not clean same-task validation.'),
    ('Class imbalance', 'EDA does not automatically prescribe SMOTE/CNB. Any imbalance treatment is a controlled modelling-stage sensitivity analysis.'),
    ('Sarcasm', 'TweetEval sarcasm-indicated test N remains exploratory; scarcity is not repaired by changing cues or pooling datasets.'),
    ('Reference group', 'reference-unmarked is an operational comparison group, not a privileged/protected-class analogue.')
]
for k,v in decisions:
    print(f'- {k}: {v}')

- Primary dataset: TweetEval only for training/model selection/final held-out test.
- Test-set discipline: No model, feature, threshold or subgroup definition may be selected from TweetEval test performance.
- Linguistic tagging: Frozen TweetEval rules are reused unchanged on external datasets.
- Subgroup eligibility: N >= 30 is assessed inside the relevant dataset/split; counts are never pooled across corpora.
- Sentiment140: Binary external robustness only; fixed 100k balanced unique-text sample; text_eval removes source-like emoticons.
- SemEval-2014: Domain/task-shift stress test only; not clean same-task validation.
- Class imbalance: EDA does not automatically prescribe SMOTE/CNB. Any imbalance treatment is a controlled modelling-stage sensitivity analysis.
- Sarcasm: TweetEval sarcasm-indicated test N remains exploratory; scarcity is not repaired by changing cues or pooling datasets.
- Reference group: reference-unmarked is an operational comparison group, not a privileged/prote

## Cell 10 — Save core-dataset freeze

In [ ]:
role_matrix.to_csv(OUTDIR/'core_dataset_role_matrix.csv', index=False)
comparability.to_csv(OUTDIR/'core_dataset_comparability_matrix.csv', index=False)
coverage.to_csv(OUTDIR/'core_subgroup_coverage_matrix.csv', index=False)
transfer_decision.to_csv(OUTDIR/'core_subgroup_transfer_decision.csv', index=False)

freeze = {
    'status':'CORE_DATASET_EDA_FROZEN',
    'approved_core_datasets':['TweetEval','SemEval-2014 Task 4','Sentiment140'],
    'primary_dataset':'TweetEval',
    'minimum_n_for_stable_quantitative_interpretation':MIN_N,
    'linguistic_rule_source':'TweetEval frozen configuration',
    'external_threshold_tuning':False,
    'pooling_across_datasets_for_subgroup_eligibility':False,
    'dataset_roles':{
        'TweetEval':'primary sentiment training/selection/final-test dataset',
        'Sentiment140':'binary external robustness only',
        'SemEval-2014':'domain/task-shift stress test only'
    },
    'subgroup_interpretation':{
        'TweetEval_sarcasm_indicated':'exploratory/descriptive due to N < 30 on held-out test',
        'Sentiment140':'all four focal groups meet N >= 30 in fixed 100k evaluation sample',
        'SemEval-2014':'no focal linguistic fairness-transfer claim because non-reference test coverage is inadequate and task mismatch remains'
    },
    'chapter3_use':'Methodology constraint only; later modelling outcomes must not redefine these EDA decisions.'
}

with open(OUTDIR/'core_dataset_eda_freeze.json','w',encoding='utf-8') as f:
    json.dump(freeze,f,indent=2)

print('Saved authoritative cross-dataset outputs to:', OUTDIR)
for p in sorted(OUTDIR.iterdir()):
    print(' -', p.name)

Saved authoritative cross-dataset outputs to: /content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Final_Thesis/EDA/Results/CrossDataset
 - core_dataset_comparability_matrix.csv
 - core_dataset_eda_freeze.json
 - core_dataset_role_matrix.csv
 - core_subgroup_coverage_matrix.csv
 - core_subgroup_transfer_decision.csv


## Cell 11 — Final conclusion

In [ ]:
print('CORE CROSS-DATASET EDA CONCLUSION')
print('1. TweetEval remains the only primary training/model-selection/final-test dataset.')
print('2. Sentiment140 remains binary external robustness with a fixed 100k evaluation sample.')
print('3. SemEval-2014 remains domain/task-shift stress testing, not clean same-task validation.')
print('4. Linguistic subgroup definitions remain frozen across datasets.')
print('5. TweetEval sarcasm-indicated evidence is exploratory; Sentiment140 supports all focal groups; SemEval does not support focal fairness transfer.')
print('6. No subgroup counts are pooled across corpora to create eligibility.')
print('7. No modelling or imbalance-treatment decision is made by this EDA notebook.')
print()
print('EDA_04 v2 COMPLETE — review outputs, then freeze the core-dataset EDA.')

CORE CROSS-DATASET EDA CONCLUSION
1. TweetEval remains the only primary training/model-selection/final-test dataset.
2. Sentiment140 remains binary external robustness with a fixed 100k evaluation sample.
3. SemEval-2014 remains domain/task-shift stress testing, not clean same-task validation.
4. Linguistic subgroup definitions remain frozen across datasets.
5. TweetEval sarcasm-indicated evidence is exploratory; Sentiment140 supports all focal groups; SemEval does not support focal fairness transfer.
6. No subgroup counts are pooled across corpora to create eligibility.
7. No modelling or imbalance-treatment decision is made by this EDA notebook.

EDA_04 v2 COMPLETE — review outputs, then freeze the core-dataset EDA.
